# Convolution forward and backward in pure numpy

This notebook writes a 2-D convolution layer and a max-pool layer with explicit loops, derives their backward passes, and checks every gradient against finite differences.

It matches these sections of notes.md:

- §1.2 (a 3×3 kernel over a 5×5 input, by hand): both worked examples, the all-ones kernel on a ramp and the vertical-edge kernel on a bar, recomputed.
- §1.3 (output shape): with no padding and stride 1, $H_{\text{out}} = H - k + 1$.
- §1.4 (pooling): max-pool forward, and a backward pass that sends each gradient only to the position of the maximum.
- §1.5 and §1.6 (why CNNs work on images): a one-conv-layer classifier trained on synthetic 8×8 images of horizontal and vertical bars.

The backward pass of a convolution is itself a convolution: $\partial L/\partial x$ is a "full" convolution of $\partial L/\partial y$ with the kernel rotated by 180 degrees, and $\partial L/\partial W$ is a cross-correlation of $x$ with $\partial L/\partial y$. The code does not rely on that identity; it derives each gradient entry from the forward loop and verifies it numerically.

Run: `uv run jupyter execute ml/04_deep_learning/cnn_scratch.ipynb`

## Setup

Each demo creates its own seeded generator (`np.random.default_rng` with a fixed seed), so every cell's output is reproducible on its own.

In [1]:
import numpy as np

np.random.seed(42)

## 1. conv2d forward, checked on the two examples from notes §1.2

`conv2d_forward` uses valid padding and stride 1: each output pixel is the dot product of a kernel with the input patch under it, plus a bias. The cell runs both examples from notes §1.2. First, a $3 \times 3$ all-ones kernel over the $5 \times 5$ ramp $0, \dots, 24$ sums each patch; expected output rows $(54, 63, 72)$, $(99, 108, 117)$, $(144, 153, 162)$. Second, the kernel with columns $(1, 0, -1)$ computes (left column of the patch) minus (right column) over an image with a bright middle column; expected every output row to be $(-3, 0, 3)$.

In [2]:
def conv2d_forward(x, W, b):
    '''
    x: (N, C_in, H, W_in)          input images
    W: (C_out, C_in, KH, KW)       kernels
    b: (C_out,)                    bias per output channel

    out: (N, C_out, H_out, W_out)  with H_out = H - KH + 1 (no padding, stride 1)

    out[n, co, i, j] = b[co] + sum_{ci, u, v} x[n, ci, i+u, j+v] * W[co, ci, u, v]
    i.e. each output pixel is a dot product between a kernel and the input
    patch whose top-left corner is at (i, j).
    '''
    N, C_in, H, Wi = x.shape
    C_out, C_in2, KH, KW = W.shape
    assert C_in == C_in2
    H_out, W_out = H - KH + 1, Wi - KW + 1
    out = np.zeros((N, C_out, H_out, W_out))
    for n in range(N):
        for co in range(C_out):
            for i in range(H_out):
                for j in range(W_out):
                    patch = x[n, :, i:i + KH, j:j + KW]   # (C_in, KH, KW)
                    out[n, co, i, j] = np.sum(patch * W[co]) + b[co]
    return out


# Example 1: 3x3 all-ones kernel over a 5x5 ramp (sums each patch).
x = np.arange(25, dtype=float).reshape(1, 1, 5, 5)   # 0..24
W = np.ones((1, 1, 3, 3))
out = conv2d_forward(x, W, np.zeros(1))
print("input:\n", x[0, 0])
print("output (each entry = sum of the 3x3 patch under it):\n", out[0, 0])
# top-left patch is [[0,1,2],[5,6,7],[10,11,12]] -> sums to 54
assert out[0, 0, 0, 0] == 54.0
assert np.array_equal(out[0, 0], [[54, 63, 72], [99, 108, 117], [144, 153, 162]])

# Example 2: vertical-edge kernel over a bright vertical bar in column 2.
x_bar = np.zeros((1, 1, 5, 5))
x_bar[0, 0, :, 2] = 1.0
W_edge = np.array([[1.0, 0.0, -1.0]] * 3).reshape(1, 1, 3, 3)
out_edge = conv2d_forward(x_bar, W_edge, np.zeros(1))
print("\nbar image:\n", x_bar[0, 0])
print("vertical-edge kernel output:\n", out_edge[0, 0])
assert np.array_equal(out_edge[0, 0], [[-3, 0, 3]] * 3)

input:
 [[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]
 [15. 16. 17. 18. 19.]
 [20. 21. 22. 23. 24.]]
output (each entry = sum of the 3x3 patch under it):
 [[ 54.  63.  72.]
 [ 99. 108. 117.]
 [144. 153. 162.]]

bar image:
 [[0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]]
vertical-edge kernel output:
 [[-3.  0.  3.]
 [-3.  0.  3.]
 [-3.  0.  3.]]


## 2. Finite-difference gradient checking

Every backward pass below is verified against central differences: $\partial f/\partial x_i \approx (f(x + \varepsilon e_i) - f(x - \varepsilon e_i)) / 2\varepsilon$, whose error shrinks like $\varepsilon^2$. It costs two function evaluations per parameter, which is slow but needs no derivation, so it is the reference to compare against. The cell checks the helper itself on $f(x) = \sum_i x_i^3$, whose gradient is $3x_i^2$; expected relative error around $10^{-10}$ or smaller.

In [3]:
def numeric_grad(f, x, eps=1e-5):
    # Central differences: df/dx_i ~= (f(x + eps e_i) - f(x - eps e_i)) / (2 eps).
    # Error O(eps^2). f takes no arguments and reads x, which is perturbed in place.
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps
        fp = f()
        x[idx] = old - eps
        fm = f()
        x[idx] = old
        grad[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad


def rel_err(a, b):
    return np.abs(a - b).max() / (np.abs(a).max() + np.abs(b).max() + 1e-12)


z = np.random.default_rng(3).standard_normal((3, 4))
e = rel_err(3 * z**2, numeric_grad(lambda: np.sum(z**3), z))
print(f"d/dz sum(z^3): max relative error vs 3 z^2 = {e:.2e}")
assert e < 1e-8

d/dz sum(z^3): max relative error vs 3 z^2 = 6.27e-12


## 3. conv2d backward, gradient-checked

Each forward multiply-add `out[n,co,i,j] += x[n,ci,i+u,j+v] * W[co,ci,u,v]` contributes `dout[n,co,i,j] * x[...]` to the weight's gradient and `dout[n,co,i,j] * W[...]` to the input pixel's gradient, so the backward pass loops over the same indices as the forward pass and accumulates both. The bias gradient is the sum of `dout` over batch and positions. To test it, the loss is $L = \sum \text{out} \odot P$ for a fixed random array $P$, so $\partial L/\partial\,\text{out} = P$ exactly. Expected: relative errors around $10^{-11}$ for `dx`, `dW` and `db`.

In [4]:
def conv2d_backward(x, W, dout):
    '''
    dout: (N, C_out, H_out, W_out) = dL/dout

    Forward:  out[n,co,i,j] = b[co] + sum_{ci,u,v} x[n,ci,i+u,j+v] * W[co,ci,u,v]

    Chain rule, term by term:
      dL/dW[co,ci,u,v] = sum_{n,i,j} dout[n,co,i,j] * x[n,ci,i+u,j+v]
        (each weight is used at every output position, so sum over all of
         them; this is x cross-correlated with dout)
      dL/db[co]        = sum_{n,i,j} dout[n,co,i,j]
      dL/dx[n,ci,p,q]  = sum over every (co,i,j,u,v) with p=i+u, q=j+v of
                         dout[n,co,i,j] * W[co,ci,u,v]
        (an input pixel contributes to every output pixel whose window
         covers it; this is the "full conv with the flipped kernel")
    All three are accumulated inside the forward loop structure: the chain
    rule applied to each multiply-add.
    '''
    N, C_in, H, Wi = x.shape
    C_out, _, KH, KW = W.shape
    _, _, H_out, W_out = dout.shape

    dx = np.zeros_like(x)
    dW = np.zeros_like(W)
    db = dout.sum(axis=(0, 2, 3))                          # (C_out,)

    for n in range(N):
        for co in range(C_out):
            for i in range(H_out):
                for j in range(W_out):
                    g = dout[n, co, i, j]                  # scalar
                    # forward used x[n, :, i:i+KH, j:j+KW] * W[co]
                    dW[co] += g * x[n, :, i:i + KH, j:j + KW]
                    dx[n, :, i:i + KH, j:j + KW] += g * W[co]
    return dx, dW, db


print("Gradient check: conv2d backward vs finite differences")
rng = np.random.default_rng(0)
x = rng.standard_normal((2, 3, 5, 5))    # (N=2, C_in=3, 5, 5)
W = rng.standard_normal((4, 3, 3, 3))    # (C_out=4, C_in=3, 3, 3)
b = rng.standard_normal(4)
# A fixed random projection makes the loss a scalar that depends on every output.
P = rng.standard_normal((2, 4, 3, 3))    # same shape as out


def loss():
    return np.sum(conv2d_forward(x, W, b) * P)


dout = P                                  # dL/dout = P since L = sum(out*P)
dx, dW, db = conv2d_backward(x, W, dout)

for name, analytic, var in [("dx", dx, x), ("dW", dW, W), ("db", db, b)]:
    numeric = numeric_grad(loss, var)
    e = rel_err(analytic, numeric)
    print(f"  {name}: max relative error = {e:.2e}")
    assert e < 1e-6, f"{name} gradient check FAILED"
print("  conv2d backward is correct.")

Gradient check: conv2d backward vs finite differences
  dx: max relative error = 4.19e-11
  dW: max relative error = 4.56e-11
  db: max relative error = 3.69e-11
  conv2d backward is correct.


## 4. Max pooling 2×2, stride 2

The forward pass takes the maximum of each non-overlapping $2 \times 2$ window and records where it was. Locally the max equals its largest input, so its derivative is 1 for that input and 0 for the other three: the backward pass sends each upstream gradient to the recorded position only (notes §1.4). The cell first prints a $4 \times 4$ example, where each gradient of 1 should appear at the position of its window's maximum, then runs the finite-difference check (expected relative error around $10^{-12}$).

In [5]:
def maxpool_forward(x, size=2):
    '''
    x: (N, C, H, W) -> out: (N, C, H//size, W//size)
    Also returns the argmax positions so backward knows where each max came from.
    '''
    N, C, H, W = x.shape
    H_out, W_out = H // size, W // size
    out = np.zeros((N, C, H_out, W_out))
    argmax = np.zeros((N, C, H_out, W_out, 2), dtype=int)  # (row, col) of max
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    window = x[n, c, i*size:(i+1)*size, j*size:(j+1)*size]
                    out[n, c, i, j] = window.max()
                    r, cidx = np.unravel_index(window.argmax(), window.shape)
                    argmax[n, c, i, j] = (i*size + r, j*size + cidx)
    return out, argmax


def maxpool_backward(dout, argmax, x_shape, size=2):
    # The max has derivative 1 with respect to its largest input and 0 for
    # the others, so each gradient goes only to the argmax position.
    dx = np.zeros(x_shape)
    N, C, H_out, W_out = dout.shape
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    r, cidx = argmax[n, c, i, j]
                    dx[n, c, r, cidx] += dout[n, c, i, j]
    return dx


x_small = np.array([[1, 3, 2, 0],
                    [4, 2, 1, 5],
                    [0, 1, 7, 2],
                    [6, 2, 3, 1]], dtype=float).reshape(1, 1, 4, 4)
pooled, am = maxpool_forward(x_small)
routed = maxpool_backward(np.ones_like(pooled), am, x_small.shape)
print("input:\n", x_small[0, 0])
print("max-pooled:\n", pooled[0, 0])
print("gradient of sum(pooled) w.r.t. input (1 at each window's max):\n", routed[0, 0])
assert np.array_equal(pooled[0, 0], [[4, 5], [6, 7]])

print("\nGradient check: maxpool backward vs finite differences")
rng = np.random.default_rng(1)
x = rng.standard_normal((2, 2, 4, 4))
P = rng.standard_normal((2, 2, 2, 2))


def loss():
    out, _ = maxpool_forward(x)
    return np.sum(out * P)


out, argmax = maxpool_forward(x)
dx = maxpool_backward(P, argmax, x.shape)
numeric = numeric_grad(loss, x)
e = rel_err(dx, numeric)
print(f"  dx: max relative error = {e:.2e}")
assert e < 1e-6
print("  maxpool backward is correct.")

input:
 [[1. 3. 2. 0.]
 [4. 2. 1. 5.]
 [0. 1. 7. 2.]
 [6. 2. 3. 1.]]
max-pooled:
 [[4. 5.]
 [6. 7.]]
gradient of sum(pooled) w.r.t. input (1 at each window's max):
 [[0. 0. 0. 0.]
 [1. 0. 0. 1.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]

Gradient check: maxpool backward vs finite differences
  dx: max relative error = 4.07e-12
  maxpool backward is correct.


## 5. End-to-end: a one-conv-layer classifier on synthetic bars

The data are 8×8 images with small Gaussian noise and one bright line at a random position: a row for class 0 (horizontal bar) and a column for class 1 (vertical bar). The model is conv 3×3 with 4 output channels, ReLU, max-pool 2×2, then a linear layer to 2 logits with softmax cross-entropy, 114 parameters in total ($36 + 4$ in the conv layer, $72 + 2$ in the linear layer). The training loop chains the backward passes written above, newest layer first, and takes 60 SGD steps on batches of 32. A single oriented-edge kernel separates the two classes, so expect batch accuracy to reach 1.00 and the loss to fall from about 0.74 to below half of its starting value (the assert).

In [6]:
def make_bars(n, rng):
    # 8x8 images: class 0 = one horizontal bar, class 1 = one vertical bar,
    # at a random position, plus noise.
    X = rng.standard_normal((n, 1, 8, 8)) * 0.1
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        pos = rng.integers(1, 7)
        if y[i] == 0:
            X[i, 0, pos, :] += 1.0     # horizontal bar (a bright row)
        else:
            X[i, 0, :, pos] += 1.0     # vertical bar (a bright column)
    return X, y


print("End-to-end: 1-conv-layer classifier on 8x8 bar images")
rng = np.random.default_rng(42)
Xtr, ytr = make_bars(256, rng)

# Architecture:
#   conv 3x3 (1 -> 4 channels)   (N,1,8,8) -> (N,4,6,6)
#   ReLU
#   maxpool 2x2                  (N,4,6,6) -> (N,4,3,3)
#   flatten                      (N, 36)
#   linear -> 2 logits           (N, 2)
#   softmax cross-entropy
C_out, K = 4, 3
Wc = rng.standard_normal((C_out, 1, K, K)) * 0.3
bc = np.zeros(C_out)
Wl = rng.standard_normal((C_out * 3 * 3, 2)) * 0.1
bl = np.zeros(2)
lr, batch, steps = 0.1, 32, 60
print(f"  parameters: {Wc.size + bc.size + Wl.size + bl.size}")

losses = []
for step in range(steps):
    idx = rng.integers(0, len(Xtr), size=batch)
    x, y = Xtr[idx], ytr[idx]
    N = len(x)

    # ---- forward ----
    z1 = conv2d_forward(x, Wc, bc)                # (N,4,6,6)
    a1 = np.maximum(z1, 0)                        # ReLU
    p1, argmax = maxpool_forward(a1)              # (N,4,3,3)
    flat = p1.reshape(N, -1)                      # (N,36)
    logits = flat @ Wl + bl                       # (N,2)
    probs = np.exp(logits - logits.max(1, keepdims=True))
    probs /= probs.sum(1, keepdims=True)
    loss = -np.log(probs[np.arange(N), y] + 1e-12).mean()
    acc = (probs.argmax(1) == y).mean()
    losses.append(loss)

    # ---- backward (chain rule, layer by layer, newest first) ----
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1.0               # softmax+CE gradient
    dlogits /= N                                  # (N,2)
    dWl = flat.T @ dlogits                        # (36,2)
    dbl = dlogits.sum(0)                          # (2,)
    dflat = dlogits @ Wl.T                        # (N,36)
    dp1 = dflat.reshape(p1.shape)                 # (N,4,3,3)
    da1 = maxpool_backward(dp1, argmax, a1.shape) # (N,4,6,6)
    dz1 = da1 * (z1 > 0)                          # ReLU gate
    _, dWc, dbc = conv2d_backward(x, Wc, dz1)

    # ---- SGD ----
    Wc -= lr * dWc; bc -= lr * dbc
    Wl -= lr * dWl; bl -= lr * dbl

    if step % 10 == 0 or step == steps - 1:
        print(f"  step {step:3d}  loss {loss:.4f}  batch acc {acc:.2f}")

assert losses[-1] < 0.5 * losses[0], "loss did not decrease enough"
print(f"  loss went {losses[0]:.3f} -> {losses[-1]:.3f}.")
print("All checks passed.")

End-to-end: 1-conv-layer classifier on 8x8 bar images
  parameters: 114
  step   0  loss 0.7394  batch acc 0.47


  step  10  loss 0.6720  batch acc 0.66


  step  20  loss 0.5974  batch acc 0.94


  step  30  loss 0.4864  batch acc 1.00


  step  40  loss 0.3690  batch acc 1.00


  step  50  loss 0.2525  batch acc 1.00


  step  59  loss 0.1904  batch acc 1.00
  loss went 0.739 -> 0.190.
All checks passed.
